# Apriori Algorithm: Frequent Itemsets and Association Rules

This notebook contains 1 practical(s). Run the cells from top to bottom. Run the first code cell once to install the required modules.

In [ ]:
# Install the required modules (safe to run even if they are already installed)
%pip install -q pandas numpy matplotlib

## Practical 1: Apriori Algorithm: Frequent Itemsets and Association Rules (Market Basket Analysis)

**Aim:** To apply the Apriori algorithm on a market-basket dataset to find frequent itemsets and generate association rules, and to visualise the support of the frequent itemsets.

**Objective:** To find itemsets whose support is at least a minimum support (0.6); to generate rules with confidence at least 0.7; to filter rules with lift greater than 1; to plot support of frequent itemsets.

**Theory:** Apriori finds frequent itemsets level by level, using the property that every subset of a frequent itemset must itself be frequent.

- **Support** of an itemset = fraction of transactions containing it.
- **Confidence** of A -> B = support(A and B) / support(A).
- **Lift** of A -> B = confidence(A -> B) / support(B). Lift > 1 means A and B occur together more often than by chance.

This notebook implements Apriori directly with pandas and itertools, so it needs no extra library (no mlxtend).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from itertools import combinations

# Sample market-basket dataset (1 = item bought in that transaction)
data = {
    "Milk":   [1, 1, 0, 1, 1],
    "Bread":  [1, 1, 1, 1, 0],
    "Butter": [0, 1, 1, 1, 1],
    "Jam":    [1, 0, 0, 1, 0],
    "Cheese": [0, 1, 0, 1, 1],
}
df = pd.DataFrame(data).astype(bool)
print(df)

def support(itemset):
    return df[list(itemset)].all(axis=1).mean()

def apriori(df, min_support):
    items = list(df.columns)
    current = [frozenset([i]) for i in items if support([i]) >= min_support]
    result = []
    k = 1
    while current:
        result += [(support(s), s) for s in current]
        k += 1
        candidates = {a | b for a in current for b in current if len(a | b) == k}
        # prune: all (k-1)-subsets must be frequent
        frequent_prev = set(current)
        candidates = [c for c in candidates
                      if all(frozenset(sub) in frequent_prev for sub in combinations(c, k - 1))]
        current = [c for c in candidates if support(c) >= min_support]
    out = pd.DataFrame(result, columns=["support", "itemsets"])
    out = out.assign(key=out["itemsets"].apply(lambda x: (len(x), sorted(x))))
    out = out.sort_values("key").drop(columns="key")
    return out.reset_index(drop=True)

# Step: frequent itemsets (minimum support = 0.6)
frequent_itemsets = apriori(df, min_support=0.6)
print("\nFrequent itemsets:")
print(frequent_itemsets)

# Step: association rules (minimum confidence = 0.7)
def association_rules(freq, min_confidence):
    sup = {s: v for v, s in zip(freq["support"], freq["itemsets"])}
    rows = []
    for itemset in sup:
        if len(itemset) < 2:
            continue
        for r in range(1, len(itemset)):
            for ante in combinations(sorted(itemset), r):
                ante = frozenset(ante)
                cons = itemset - ante
                conf = sup[itemset] / sup[ante]
                if conf >= min_confidence:
                    rows.append({
                        "antecedents": set(ante), "consequents": set(cons),
                        "antecedent support": sup[ante], "consequent support": sup[cons],
                        "support": sup[itemset], "confidence": conf,
                        "lift": conf / sup[cons],
                    })
    return pd.DataFrame(rows)

rules = association_rules(frequent_itemsets, min_confidence=0.7)
print("\nAssociation rules (confidence >= 0.7):")
print(rules.to_string())

# Step: keep only rules with lift > 1
high_lift_rules = rules[rules["lift"] > 1]
print("\nRules with lift > 1:")
print(high_lift_rules.to_string())

# Step: visualise support of frequent itemsets
plt.figure(figsize=(8, 4))
plt.bar(frequent_itemsets["itemsets"].apply(lambda s: ", ".join(sorted(s))), frequent_itemsets["support"])
plt.xlabel("Itemsets")
plt.ylabel("Support")
plt.title("Frequent Itemsets Support")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()